# Task 01 — Data Ingestion, Schema Checks, and Missingness Handling

**Agent:** Claude (claude.ai)  
**Dataset:** New York City Airbnb Open Data 2019 (`AB_NYC_2019.csv`)  
**Prediction target:** `price` (nightly listing price — regression)  
**Output folder:** `outputs/`

---

> **Before committing:** Kernel → Restart & Clear Output

In [ ]:
# ── Imports & reproducibility seed ─────────────────────────────────────────
import random
import numpy as np
import pandas as pd
from pathlib import Path

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Robust root-finding: works whether Cursor starts the kernel from the
# notebook directory OR the workspace root (group-coursework/).
def _find_project_root() -> Path:
    for candidate in [Path.cwd()] + list(Path.cwd().parents):
        if (candidate / "data" / "raw").exists():
            return candidate
    raise FileNotFoundError(
        "Cannot find project root — expected a parent folder containing data/raw/"
    )

PROJECT_ROOT = _find_project_root()
RAW_PATH     = PROJECT_ROOT / "data" / "raw" / "AB_NYC_2019.csv"
OUTPUT_DIR   = PROJECT_ROOT / "agents" / "claude" / "task_01" / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root  : {PROJECT_ROOT}")
print(f"Raw data path : {RAW_PATH}  (exists: {RAW_PATH.exists()})")
print(f"Output folder : {OUTPUT_DIR}")

## 1. Load and Inspect the Dataset

Before writing a single line of cleaning code, we need to understand what the data actually contains.

In [ ]:
df = pd.read_csv(RAW_PATH)
print(f"Shape: {df.shape}  ({df.shape[0]:,} rows, {df.shape[1]} columns)")
df.head()

In [ ]:
df.info()

In [ ]:
df.describe(include="all")

## 2. Schema Log

Recording column names, inferred dtypes, non-null counts, and what each column represents.  
Saved to `outputs/schema_log.txt`.

In [ ]:
COLUMN_DESCRIPTIONS = {
    "id":                             "Unique listing identifier — not a feature",
    "name":                           "Free-text listing title — not used in baseline modelling",
    "host_id":                        "Unique host identifier — not a feature",
    "host_name":                      "Host first name — not used in baseline modelling",
    "neighbourhood_group":            "Borough (Manhattan, Brooklyn, Queens, Bronx, Staten Island) — categorical feature",
    "neighbourhood":                  "Sub-neighbourhood — high-cardinality categorical feature",
    "latitude":                       "Listing latitude — numeric feature",
    "longitude":                      "Listing longitude — numeric feature",
    "room_type":                      "Type of space (Entire home/apt, Private room, Shared room) — categorical feature",
    "price":                          "*** TARGET *** Nightly price in USD",
    "minimum_nights":                 "Minimum booking length in nights — numeric feature, known extremes",
    "number_of_reviews":              "Total review count — numeric feature",
    "last_review":                    "Date of most recent review — missing when no reviews exist",
    "reviews_per_month":              "Avg reviews per month — missing when no reviews exist",
    "calculated_host_listings_count": "Number of listings by same host — numeric feature",
    "availability_365":               "Days available in next 365 — numeric feature",
}

schema_lines = [
    "SCHEMA LOG — AB_NYC_2019.csv",
    "=" * 60,
    f"Rows    : {len(df):,}",
    f"Columns : {len(df.columns)}",
    "",
    f"{'Column':<40} {'dtype':<12} {'non-null':>10}  Description",
    "-" * 100,
]
for col in df.columns:
    non_null = df[col].notna().sum()
    desc = COLUMN_DESCRIPTIONS.get(col, "—")
    schema_lines.append(f"{col:<40} {str(df[col].dtype):<12} {non_null:>10,}  {desc}")

schema_text = "\n".join(schema_lines)
(OUTPUT_DIR / "schema_log.txt").write_text(schema_text)
print(schema_text)

## 3. Missingness Assessment

We distinguish between two kinds of missing values:

1. **Structural missingness** — the value is missing *because of something real*. A listing with zero reviews will naturally have no `last_review` date and no `reviews_per_month`. This is not a data error — it encodes the fact that the listing is unreviewed.
2. **Random / corrupt missingness** — the value should exist but was not recorded (e.g. a listing with no `name`).

The handling strategy differs for each type.

In [ ]:
missing_counts = df.isnull().sum()
missing_pct    = (missing_counts / len(df) * 100).round(2)

missing_df = pd.DataFrame({
    "missing_count": missing_counts,
    "missing_pct":   missing_pct,
    "dtype":         df.dtypes
}).sort_values("missing_pct", ascending=False)

print(missing_df[missing_df.missing_count > 0].to_string())
missing_df.to_csv(OUTPUT_DIR / "missingness_report.csv")
print("\nSaved: outputs/missingness_report.csv")

### Missingness Decisions

**`last_review` and `reviews_per_month`** (~20% missing)  
Cross-checking confirms these are missing *only* for listings with `number_of_reviews == 0`. This is structural missingness — not a data error.  
- `last_review`: a date string. Even imputed, it would require parsing, feature engineering (days since last review), and adds complexity. We **drop this column** — limited predictive signal at baseline stage.  
- `reviews_per_month`: numerically meaningful. A listing with no reviews has an effective rate of **0**. We **fill with 0** — this preserves the information that the listing is unreviewed.

**`name`** (~0.03% missing)  
Free-text listing title. Requires NLP to use. Not a structured feature. We **drop this column**.

**`host_name`** (~0.01% missing)  
Host first name. Not predictive. We **drop this column**.

In [ ]:
# Verify: last_review and reviews_per_month are missing ONLY when number_of_reviews == 0
reviews_zero     = df["number_of_reviews"] == 0
last_rev_missing = df["last_review"].isnull()
rpm_missing      = df["reviews_per_month"].isnull()

print(f"Rows with 0 reviews                       : {reviews_zero.sum()}")
print(f"last_review missing                       : {last_rev_missing.sum()}")
print(f"reviews_per_month missing                 : {rpm_missing.sum()}")
print(f"last_review missing but reviews > 0       : {(last_rev_missing & ~reviews_zero).sum()}")
print(f"reviews_per_month missing but reviews > 0 : {(rpm_missing & ~reviews_zero).sum()}")

## 4. Data Quality: Invalid Values and Outlier Flags

Beyond missingness, we check for impossible values and extreme outliers.

In [ ]:
price_zero = (df["price"] == 0).sum()
print(f"Listings with price = $0 : {price_zero}")
print(df[df["price"] == 0][["id", "name", "room_type", "neighbourhood_group", "price"]].head(10))

### Decision: Drop price = $0

Airbnb does not permit free listings. A `price` of $0 is either a data entry error, a withdrawn listing, or a corrupt record. Including these would corrupt the regression target — the model would learn that some valid-looking listings cost nothing.  
**Action:** Drop all rows where `price == 0`.

**High prices (>$500, >$1000) are NOT removed.** Luxury Airbnb listings in Manhattan at $500–$10,000/night exist and are real. Task 01 explicitly prohibits transforming the target variable. Log-transformation and outlier treatment will be evaluated in Task 03.

In [ ]:
price_nonzero = df[df["price"] > 0]["price"]
print("price distribution (non-zero listings):")
print(price_nonzero.describe())
print(f"\nListings price > $500  : {(price_nonzero > 500).sum()}")
print(f"Listings price > $1000 : {(price_nonzero > 1000).sum()}")
print(f"Max price              : ${price_nonzero.max():,}")

print("\nminimum_nights distribution:")
print(df["minimum_nights"].describe())
print(f"Listings minimum_nights > 365 : {(df['minimum_nights'] > 365).sum()}")

In [ ]:
flag_lines = [
    "OUTLIER / SUSPICIOUS VALUE FLAGS — AB_NYC_2019",
    "=" * 60,
    "",
    "price",
    f"  Rows with price = $0       : {(df['price'] == 0).sum()}  → DROPPED (invalid — Airbnb prohibits $0 listings)",
    f"  Rows with price > $500     : {(df['price'] > 500).sum()}  → FLAGGED only — valid luxury listings exist",
    f"  Rows with price > $1000    : {(df['price'] > 1000).sum()}  → FLAGGED only",
    f"  Max price                  : ${df['price'].max():,}",
    "",
    "minimum_nights",
    f"  Rows > 365 nights          : {(df['minimum_nights'] > 365).sum()}  → FLAGGED only — long-term rentals are valid",
    f"  Max minimum_nights         : {df['minimum_nights'].max()}",
    "",
    "RATIONALE: price outliers and minimum_nights extremes are NOT removed here.",
    "Task 01 constraint: do not transform the target variable.",
    "Outlier treatment for price and minimum_nights will be evaluated in Task 03.",
]
(OUTPUT_DIR / "outlier_flags.txt").write_text("\n".join(flag_lines))
print("\n".join(flag_lines))

## 5. Column Decisions

| Column | Decision | Reason |
|--------|----------|--------|
| `id` | **Drop** | Row identifier — no predictive value; risk of model memorising IDs |
| `name` | **Drop** | Free text — requires NLP; not used in structured baseline |
| `host_id` | **Drop** | Identifier — no predictive value |
| `host_name` | **Drop** | Free text identifier — no predictive value |
| `neighbourhood_group` | **Keep** | Borough is a strong price predictor (Manhattan commands a premium) |
| `neighbourhood` | **Keep** | High-cardinality but informative; encoding deferred to Task 03 |
| `latitude` | **Keep** | Geographic location carries price signal |
| `longitude` | **Keep** | Geographic location carries price signal |
| `room_type` | **Keep** | Entire home vs private room is one of the strongest price drivers |
| `price` | **Keep** | **TARGET** |
| `minimum_nights` | **Keep** | Proxy for listing type (short vs long-stay); extreme values flagged |
| `number_of_reviews` | **Keep** | Proxy for popularity and trust |
| `last_review` | **Drop** | Date field; ~20% structurally missing; limited signal without feature engineering |
| `reviews_per_month` | **Keep** | Fill 0 for unreviewed listings — rate of 0 is meaningful |
| `calculated_host_listings_count` | **Keep** | Distinguishes professional hosts from casual ones |
| `availability_365` | **Keep** | Reflects host pricing strategy and listing demand |

## 6. Apply All Cleaning Decisions

Changes applied in order:
1. Drop identifier and free-text columns: `id`, `name`, `host_id`, `host_name`, `last_review`
2. Fill `reviews_per_month` NaN → 0
3. Drop rows where `price == 0`

No other rows dropped. No values imputed for the target. No train/test split.

In [ ]:
df_clean = df.copy()

# 1 — Drop identifier and free-text columns
COLS_TO_DROP = ["id", "name", "host_id", "host_name", "last_review"]
df_clean = df_clean.drop(columns=COLS_TO_DROP)
print(f"After dropping identifier/text columns : {df_clean.shape}")

# 2 — Fill reviews_per_month NaN → 0 (structural: no reviews = rate of 0)
n_filled = df_clean["reviews_per_month"].isnull().sum()
df_clean["reviews_per_month"] = df_clean["reviews_per_month"].fillna(0)
print(f"reviews_per_month NaNs filled with 0   : {n_filled:,} rows")

# 3 — Drop rows where price == 0 (invalid listings)
n_before = len(df_clean)
df_clean = df_clean[df_clean["price"] > 0].reset_index(drop=True)
n_dropped = n_before - len(df_clean)
print(f"Rows dropped (price = $0)              : {n_dropped}")
print(f"\nFinal shape : {df_clean.shape}")

## 7. Verification

Confirming the cleaned dataset is free of every issue we identified.

In [ ]:
print("=" * 50)
print("POST-CLEANING VERIFICATION")
print("=" * 50)

total_missing = df_clean.isnull().sum().sum()
assert total_missing == 0, f"Still has {total_missing} missing values!"
print("  PASS  No missing values")

assert (df_clean["price"] == 0).sum() == 0
print("  PASS  No price = $0 rows")

for col in ["id", "name", "host_id", "host_name", "last_review"]:
    assert col not in df_clean.columns, f"{col} still present!"
print("  PASS  All identifier / text columns removed")

assert "price" in df_clean.columns
print("  PASS  Target column 'price' present")

print(f"\nFinal missing values : {df_clean.isnull().sum().sum()}")
print(f"Final shape          : {df_clean.shape}")
print(f"Columns              : {list(df_clean.columns)}")
df_clean.dtypes

## 8. Save Output

In [ ]:
out_path = OUTPUT_DIR / "ingested.csv"
df_clean.to_csv(out_path, index=False)
print(f"Saved  : {out_path}")
print(f"Rows   : {len(df_clean):,}")
print(f"Cols   : {len(df_clean.columns)}")
df_clean.head()

## Summary of All Decisions

| Change | Scope | Justification |
|--------|-------|---------------|
| Drop `id`, `host_id` | 2 columns | Pure identifiers — no predictive value, leakage risk |
| Drop `name`, `host_name` | 2 columns | Free text requiring NLP; minor missingness; not used in structured baseline |
| Drop `last_review` | 1 column | Date string; structurally missing for ~20% of rows (zero-review listings); feature engineering deferred |
| Fill `reviews_per_month` = 0 | ~10,000 rows | Structural missingness: an unreviewed listing has a genuine review rate of 0, not an unknown value |
| Drop `price == 0` rows | ~11 rows | Impossible value — Airbnb prohibits $0 listings; these are corrupt/withdrawn entries |
| Retain `price` outliers (>$500) | 0 rows dropped | Task 01 constraint: do not transform the target variable |
| Retain `minimum_nights` extremes | 0 rows dropped | Long-term rentals are valid Airbnb listings; outlier treatment deferred to Task 03 |

**What was explicitly NOT done here:**
- No train/test split — prevents any leakage from the splitting step into ingestion
- No scaling or encoding — these must be fit on training data only (Task 03)
- No log-transform of `price` — explicit Task 01 constraint
- No imputation using any price-derived statistic — would be target leakage